# "THE POUR IS RIGHT" - Wine Pricer Capstone

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 4: Neural Networks and LLMs

Today we go from traditional ML to neural networks to large language models. This follows `week6/day4.ipynb`.

The score to beat from day 3 is XGBoost at \$11.67.

## Costs

The neural networks run on your laptop and are free. The four LLMs each price the same 200 test wines:

| Model | Rough cost for 200 wines |
|---|---|
| GPT-4.1 Nano (OpenAI) | under 1 cent |
| GPT-OSS 20B (Groq) | under 1 cent |
| Qwen 3.8 27B (Hugging Face) | a few cents at most |
| Claude Sonnet 5 (Anthropic) | about 8 cents |

Each model has its own cell, so skip any you don't want to pay for.

In [ ]:
# imports

import os
import csv
import numpy as np
from dotenv import load_dotenv
from huggingface_hub import HfApi
from tqdm.notebook import tqdm
from litellm import completion
from openai import OpenAI
import anthropic
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from pricer.evaluator import evaluate
from pricer.items import Item
from pricer.deep_neural_network import DeepNeuralNetworkRunner

load_dotenv(override=True)

In [ ]:
username = HfApi().whoami()["name"]
dataset = f"{username}/wines_lite"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

# Optional: a different kind of neural network

Ed priced 100 products himself, to see how a human compares with the models. This section does the same for wine. It is optional: skip to "A vanilla Neural Network" if you don't want to do it.

To take part:

1. Run the next cell. It writes the first 100 test wines to `human_in.csv`, each with a price of 0.
2. Open `human_in.csv` in a spreadsheet, replace each 0 with your guess in dollars, and save it as `human_out.csv` in this folder.
3. Run the cells after it.

In [ ]:
# Write the first 100 test wines to a CSV

with open('human_in.csv', 'w', encoding="utf-8", newline="") as csvfile:
    writer = csv.writer(csvfile)
    for t in test[:100]:
        writer.writerow([t.summary, 0])

In [ ]:
# Read your guesses back in, if you made them

human_predictions = []

if os.path.exists('human_out.csv'):
    with open('human_out.csv', 'r', encoding="utf-8") as csvfile:
        reader = csv.reader(csvfile)
        for row in reader:
            human_predictions.append(float(row[1]))
    print(f"Read {len(human_predictions)} human guesses")
else:
    print("No human_out.csv found - skipping the human baseline")

In [ ]:
def human_pricer(item):
    idx = test.index(item)
    return human_predictions[idx]

if len(human_predictions) >= 100:
    evaluate(human_pricer, test, size=100)

# A vanilla Neural Network

This is Ed's 8-layer network, built from scratch in PyTorch. Like yesterday's models it sees each wine as a bag of words: 5,000 numbers, each 1 if a word is present and 0 if not.

Each layer multiplies its inputs by a set of weights and passes the result on. Training adjusts those weights, a little at a time, to shrink the error.

Ed trained for 2 epochs (passes through the data) on 800,000 products. We have 20,000 wines, so we use 3. With this little data the network starts memorising the training wines after that, and gets worse on new ones. Expect about \$12.

In [ ]:
# Prepare our documents and prices

y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [ ]:
# Use the HashingVectorizer for a Bag of Words model
# Using binary=True makes "one-hot vectors": 1 if the word is there, 0 if not

np.random.seed(42)
vectorizer = HashingVectorizer(n_features=5000, stop_words='english', binary=True)
X = vectorizer.fit_transform(documents)

In [ ]:
# Define the neural network - here is Pytorch code to create a 8 layer neural network

class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 64)
        self.layer4 = nn.Linear(64, 64)
        self.layer5 = nn.Linear(64, 64)
        self.layer6 = nn.Linear(64, 64)
        self.layer7 = nn.Linear(64, 64)
        self.layer8 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.relu(self.layer3(output2))
        output4 = self.relu(self.layer4(output3))
        output5 = self.relu(self.layer5(output4))
        output6 = self.relu(self.layer6(output5))
        output7 = self.relu(self.layer7(output6))
        output8 = self.layer8(output7)
        return output8

In [ ]:
torch.manual_seed(42)

# Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train_tensor, y_train_tensor, test_size=0.05, random_state=42)

# Create the loader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Initialize the model
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

In [ ]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Number of trainable parameters: {trainable_params:,}")

In [ ]:
# Define loss function and optimizer

loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# We will do 3 complete runs through the data

EPOCHS = 3

for epoch in range(EPOCHS):
    model.train()
    for batch_X, batch_y in tqdm(train_loader):
        optimizer.zero_grad()

        # The next 4 lines are the 4 stages of training: forward pass, loss calculation, backward pass, optimize

        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)

    print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss: {val_loss.item():.3f}')

In [ ]:
def neural_network(item):
    model.eval()
    with torch.no_grad():
        vector = vectorizer.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray())
        result = model(vector)[0].item()
    return max(0, result)

In [ ]:
evaluate(neural_network, test)

# A better Neural Network

Ed's optional "redemption" network (`week6/redemption_train.ipynb`) fixed the weaknesses of the vanilla one. His had 8 residual blocks of 4,096 and took 4 hours on a GPU with 800,000 products. Ours, in `pricer/deep_neural_network.py`, keeps every idea but is sized for 20,000 wines on a laptop: 2 blocks of 256, about 40 seconds to train.

The ideas, in order of how much they matter here:

- **Predict the logarithm of the price.** Yesterday every model under-priced expensive wines. In log terms, the step from \$10 to \$20 is the same size as the step from \$100 to \$200, so the network cares about being 20% off and not about being \$20 off. That stops the many cheap wines from drowning out the few expensive ones.
- **L1 loss instead of MSE.** MSE squares each error, so one wild miss dominates training. L1 uses the plain size of the error, which is also what we measure.
- **Dropout.** During training, a random 20% of the values are switched off at each step. The network can't lean on any single word, which reduces memorising.
- **Residual blocks.** Each block adds its input back to its output, which makes deeper networks easier to train.
- **A falling learning rate.** The adjustments start large and shrink towards zero, so training settles instead of bouncing around.

Expect about \$11.20, a little ahead of XGBoost.

In [ ]:
runner = DeepNeuralNetworkRunner(train, val)
runner.setup()

In [ ]:
runner.train(epochs=6)

In [ ]:
def deep_neural_network(item):
    return runner.inference(item)

evaluate(deep_neural_network, test)

In [ ]:
# Save the weights - we will need this model again in week 8

runner.save('deep_neural_network.pth')

# And now - to the frontier!

How well do large language models price wine with no training on our data at all? They have read a great deal about wine, so they may already know what a Chambertin or a Napa Cabernet tends to cost.

Every model gets the same short prompt and the same 200 test wines.

In [ ]:
def messages_for(item):
    message = f"Estimate the price of this bottle of wine in US dollars. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [ ]:
print(messages_for(test[0])[0]["content"])
print()
print(f"Actual price: ${test[0].price}")

## GPT-4.1 Nano

OpenAI's cheapest model. This is the one we fine-tune on day 5, so this score is its "before".

In [ ]:
def gpt_4__1_nano(item):
    response = completion(model="openai/gpt-4.1-nano", messages=messages_for(item))
    return response.choices[0].message.content

In [ ]:
gpt_4__1_nano(test[0])

In [ ]:
evaluate(gpt_4__1_nano, test)

## GPT-OSS 20B on Groq

The open model that wrote our summaries on day 2.

In [ ]:
def gpt_oss_20b(item):
    response = completion(model="groq/openai/gpt-oss-20b", messages=messages_for(item), reasoning_effort="low")
    return response.choices[0].message.content

In [ ]:
evaluate(gpt_oss_20b, test)

## Qwen 3.8 27B on Hugging Face

A fully open-source model (Apache-2.0 licence), run through Hugging Face's Inference Providers with the `HF_TOKEN` we already have. Hugging Face offers an OpenAI-style endpoint, so we use the OpenAI client pointed at it. Usage is billed to your Hugging Face account.

Qwen 3.8 "thinks" before it answers. Left alone it used its whole token allowance thinking and never gave a price, so we set `reasoning_effort="low"`, as Ed does with other reasoning models.

In [ ]:
def qwen_3__8_27b(item):
    response = completion(model="groq/qwen/qwen3.8-27b", messages=messages_for(item), reasoning_effort="none")
    return response.choices[0].message.content


In [ ]:
evaluate(qwen_3__8_27b, test)

## Claude Sonnet 5

One strong frontier model as the benchmark. About 8 cents for the 200 wines.

It is called through Anthropic's own SDK instead of LiteLLM. Two things about the call:

- `effort` controls how much the model thinks before answering. `low` is plenty for a one-line price, and at that setting it answers directly.
- The reply is a list of blocks, so we pick out the text block.

Claude Opus 5.5 is the stronger model and would cost about 40 cents for the same run. To try it, change the model to `claude-opus-5-5`.

In [ ]:
claude = anthropic.Anthropic()

def claude_sonnet_5(item):
    response = claude.messages.create(
        model="claude-sonnet-5",
        max_tokens=16000,
        messages=messages_for(item),
        extra_body={"output_config": {"effort": "low"}},
    )
    if response.stop_reason == "refusal":
        return "0"
    return next((block.text for block in response.content if block.type == "text"), "0")

In [ ]:
claude_sonnet_5(test[0])

In [ ]:
evaluate(claude_sonnet_5, test)

## Results so far

Fill in the errors from your own runs for the four LLMs (they are set to 0 below), then run the cell.

In [ ]:
import plotly.graph_objects as go

results = [
    ("Constant", "gray", 21.71),
    ("Linear Regression", "gray", 16.46),
    ("NLP + LR", "gray", 13.50),
    ("Random Forest", "gray", 12.01),
    ("XGBoost", "gray", 11.67),
    ("Neural Network", "orange", 12.02),
    ("Deep Neural Network", "orange", 11.23),
    ("GPT 4.1 Nano", "slateblue", 15.10),
    ("GPT-OSS 20B", "slateblue", 32.79),
    ("Qwen 3.8 27B", "slateblue", 14.42),
    ("Claude Sonnet 5", "slateblue", 9.06),
]

results = [result for result in results if result[2] > 0]
labels, colors, values = zip(*results)

fig = go.Figure(go.Bar(x=labels, y=values, marker_color=colors, text=[f"${v:.2f}" for v in values], textposition="outside"))

fig.update_layout(
    title="Wine pricer - average error by model (lower is better)",
    yaxis=dict(range=[0, max(values) * 1.15], title="Error ($)"),
    xaxis=dict(tickangle=-45),
    width=1000,
    height=700
)

fig.show()